
# Sistemas Basados en Reglas  

Este notebook muestra ejemplos prácticos de **sistemas expertos basados en reglas**,
aplicados al área de **seguridad e higiene laboral**.  

Incluye:  
1. Reglas condicionales simples (IF-THEN).  
2. Reglas con múltiples condiciones (IF-AND-OR).  
3. Reglas con prioridad o jerarquía.  
4. Sistema de producción (motor de inferencia + base de reglas).  

In [ ]:
# ====================================
# Clase 3: Sistemas basados en reglas
# Ejemplos de seguridad e higiene
# ====================================

# --- 1. Reglas condicionales simples ---
print("=== Reglas condicionales simples ===")
temperatura = 39
# Definimos una variable llamada "temperatura".
# En este caso, el valor es 39°C (podría representar la temperatura corporal de un trabajador).

if temperatura > 38:
    print("Riesgo: posible golpe de calor.")
else:
    print("Condiciones normales.")

=== Reglas condicionales simples ===
Riesgo: posible golpe de calor.


In [ ]:
# --- 2. Reglas con múltiples condiciones (AND / OR) ---
print("\n=== Reglas con múltiples condiciones ===")
casco = False # Variable que indica si el trabajador usa casco de seguridad.
# False = no tiene casco, True = sí tiene casco.
zona_construccion = True

if not casco and zona_construccion:# Regla 1: SI el trabajador NO usa casco Y está en zona de construcción,
    # ENTONCES hay riesgo ALTO.
    print("Riesgo alto: trabajador sin casco en zona de construcción.")
elif not casco:
    print("Riesgo moderado: trabajador sin casco, pero fuera de zona de construcción.")
else:
    print("Condiciones seguras.")


=== Reglas con múltiples condiciones ===
Riesgo alto: trabajador sin casco en zona de construcción.


In [ ]:
# --- 3. Reglas con prioridad o jerarquía ---
print("\n=== Reglas con prioridad ===")
situaciones = {"fuego": True, "ruido": True}# Definimos un diccionario llamado "situaciones".

if situaciones["fuego"]:# Primera regla: SI hay fuego,
    # ENTONCES la acción más importante es evacuar inmediatamente.
    # Esta regla tiene máxima prioridad.
    print("PRIORIDAD 1 → Evacuar inmediatamente.")
elif situaciones["ruido"]:
    print("PRIORIDAD 2 → Usar protección auditiva.")# Segunda regla: SI hay ruido (y no hay fuego),
    # ENTONCES la acción es usar protección auditiva.
    # Esta regla tiene menor prioridad que la del fuego.


=== Reglas con prioridad ===
PRIORIDAD 1 → Evacuar inmediatamente.


¿Qué es lambda en Python?

lambda define una función anónima (una función sin nombre).

Se usa cuando querés una función rápida y simple en una sola línea.

Es equivalente a definir con def, pero más corto.

In [ ]:
# --- 4. Sistema de producción ---
print("\n=== Sistema de producción ===")

# Base de reglas
# Primera regla: si NO hay casco en los hechos y SÍ hay 'zona_construccion'
    # entonces se agrega el hecho "riesgo_accidente".
reglas = [
    {"condicion": lambda h: "casco" not in h and "zona_construccion" in h,
     "accion": lambda h: h.add("riesgo_accidente")},# Segunda regla: si ya se detectó "riesgo_accidente",
    # entonces se agrega el hecho "notificar_supervisor".
    {"condicion": lambda h: "riesgo_accidente" in h,
     "accion": lambda h: h.add("notificar_supervisor")},
]
# Hechos inicialesel trabajador está en una zona de construcción
hechos = {"zona_construccion"}
print("Hechos iniciales:", hechos)

# Motor de inferencia (forward simple)
cambio = True# Bandera para controlar si hubo cambios en los hechos
while cambio:
    cambio = False
    for regla in reglas:# Recorremos cada regla definida
        if regla["condicion"](hechos): # Si la condición se cumple
            antes = hechos.copy()  # Guardamos copia para comparar cambios
            regla["accion"](hechos)# Ejecutamos la acción asociada a la regla
            if hechos != antes: # Si los hechos cambiaron (hubo una nueva inferencia)
                print("→ Nueva inferencia:", hechos)
                cambio = True# Se activa la bandera para seguir inferiendo

print("Hechos finales:", hechos)


=== Sistema de producción ===
Hechos iniciales: {'zona_construccion'}
→ Nueva inferencia: {'riesgo_accidente', 'zona_construccion'}
→ Nueva inferencia: {'notificar_supervisor', 'riesgo_accidente', 'zona_construccion'}
Hechos finales: {'notificar_supervisor', 'riesgo_accidente', 'zona_construccion'}


Base de reglas: está formada por un conjunto de condiciones (condicion) y acciones (accion).

Hechos iniciales: representan lo que sabemos al comienzo (en este caso, que el trabajador está en una zona de construcción).

Motor de inferencia (forward chaining): recorre las reglas y aplica aquellas cuyas condiciones se cumplen, agregando nuevos hechos.

Encadenamiento: el proceso continúa hasta que no se agregan más hechos.

En el ejemplo:

Parte de {"zona_construccion"}.

Se activa la primera regla → agrega "riesgo_accidente".

Como ahora existe "riesgo_accidente", se activa la segunda regla → agrega "notificar_supervisor".

Resultado final: {"zona_construccion", "riesgo_accidente", "notificar_supervisor"}.

Esto ya es un mini sistema experto que muestra cómo un sistema de producción genera nuevo conocimiento a partir de hechos iniciales.

In [ ]:
# ============================
# Ejemplo de encadenamiento hacia adelante y hacia atrás
# ============================

# --- Encadenamiento hacia adelante ---
def forward_chaining(hechos, reglas):
    # Convierte la lista de hechos iniciales en un conjunto para búsquedas rápidas
    hechos = set(hechos)
    # Variable de control para saber si en la última iteración se aplicó alguna regla
    aplicado = True
    # Lista para guardar los pasos o reglas que se van aplicando
    pasos = []
    # Mientras se siga aplicando alguna regla, el bucle continúa
    while aplicado:
        aplicado = False  # al inicio de cada ciclo se asume que no se aplicó nada
        # Recorre todas las reglas disponibles
        for condicion, conclusion in reglas:
            # Si todos los hechos de la condición están presentes
            # y la conclusión aún no se agregó a los hechos
            if condicion.issubset(hechos) and conclusion not in hechos:
                # Agrega la conclusión como un nuevo hecho
                hechos.add(conclusion)
                # Guarda una descripción de la regla aplicada
                pasos.append(f"Se aplicó regla {condicion} -> {conclusion}")
                # Marca que se aplicó al menos una regla en esta iteración
                aplicado = True
    # Devuelve los hechos finales y la lista de pasos aplicados
    return hechos, pasos

# --- Encadenamiento hacia atrás ---
def backward_chaining(meta, hechos, reglas):
    # Caso base: si la meta ya está entre los hechos conocidos, se cumple
    if meta in hechos:
        return True, [f"La meta {meta} ya es un hecho."]

    # Si no está, se buscan reglas cuya conclusión sea la meta
    for condicion, conclusion in reglas:
        if conclusion == meta:
            # Se necesita demostrar cada hecho de la condición
            pasos = [f"Para demostrar {meta}, necesito {condicion}"]
            ok = True  # bandera para saber si todas las sub-metas se logran
            for h in condicion:
                # Llamada recursiva para probar cada hecho necesario
                subok, subpasos = backward_chaining(h, hechos, reglas)
                pasos.extend(subpasos)  # agrega los pasos de la subprueba
                if not subok:          # si alguna condición no se puede probar
                    ok = False         # la meta tampoco se puede probar
                    break
            # Si todas las condiciones se probaron con éxito
            if ok:
                return True, pasos
    # Si ninguna regla permite demostrar la meta, se falla
    return False, [f"No se pudo probar {meta} con los hechos actuales."]

# --- Reglas y hechos del ejemplo ---
reglas = [
    # Cada regla es una tupla: (conjunto de condiciones, conclusión)
    ({"zona_construccion", "sin_casco"}, "riesgo_accidente"),
    ({"riesgo_accidente"}, "notificar_supervisor")
]

# Hechos de partida conocidos
hechos_iniciales = {"zona_construccion", "sin_casco"}

# Encadenamiento hacia adelante
hechos_finales, pasos_forward = forward_chaining(hechos_iniciales, reglas)
print("=== ENCADENAMIENTO HACIA ADELANTE ===")
print("Hechos finales:", hechos_finales)   # Muestra todos los hechos deducidos
for p in pasos_forward:
    print(p)                                # Lista las reglas aplicadas

# Encadenamiento hacia atrás
meta = "notificar_supervisor"               # Objetivo a demostrar
exito, pasos_backward = backward_chaining(meta, hechos_iniciales, reglas)
print("\n=== ENCADENAMIENTO HACIA ATRÁS ===")
print(f"¿Se pudo probar la meta '{meta}'?:", exito)  # Resultado (True/False)
for p in pasos_backward:
    print(p)                                 # Explica el razonamiento paso a paso


=== ENCADENAMIENTO HACIA ADELANTE ===
Hechos finales: {'riesgo_accidente', 'zona_construccion', 'sin_casco', 'notificar_supervisor'}
Se aplicó regla {'zona_construccion', 'sin_casco'} -> riesgo_accidente
Se aplicó regla {'riesgo_accidente'} -> notificar_supervisor

=== ENCADENAMIENTO HACIA ATRÁS ===
¿Se pudo probar la meta 'notificar_supervisor'?: True
Para demostrar notificar_supervisor, necesito {'riesgo_accidente'}
Para demostrar riesgo_accidente, necesito {'zona_construccion', 'sin_casco'}
La meta zona_construccion ya es un hecho.
La meta sin_casco ya es un hecho.


Hacia adelante:
Parte de los hechos zona_construccion y sin_casco.

Aplica la regla 1 → añade riesgo_accidente.

Aplica la regla 2 → añade notificar_supervisor.

Hacia atrás:
Parte de la meta notificar_supervisor.

Verifica que necesita riesgo_accidente.

Luego necesita zona_construccion y sin_casco, que ya están como hechos.

**EJERCICIO**
Copiar este código en una celda de Google Colab y ejecutarlo.

Modificar los hechos iniciales:

Por ejemplo, dejar solo {"zona_construccion"} (sin el hecho de que no tiene casco).

Observar cómo cambia el resultado en ambos métodos.
**Agregar una nueva regla:**

({"notificar_supervisor"}, "cerrar_zona")
y verificar cómo el encadenamiento hacia adelante agrega una nueva conclusión automáticamente.

# Sistema experto simple: Identificación de frutas
Este ejercicio aplica **reglas IF... THEN...** para reconocer frutas según sus características.

---

###  Objetivos
1. Entender cómo funcionan los **sistemas basados en reglas**.
2. Implementar reglas en Python.
3. Probar casos de entrada y analizar resultados.
4. Extender el sistema agregando nuevas reglas.

###La idea general
Entrada: color y tamaño de la fruta.

Base de conocimiento: un conjunto de reglas IF…THEN.

Salida: el nombre de la fruta que cumpla las condiciones.

###  Reglas del sistema

1. IF color = "rojo" AND tamaño = "pequeño" THEN fruta = "cereza"  
2. IF color = "amarillo" AND tamaño = "grande" THEN fruta = "banana"  
3. IF color = "naranja" AND tamaño = "mediano" THEN fruta = "mandarina"  
4. IF color = "rojo" AND tamaño = "grande" THEN fruta = "manzana"  
5. IF color = "verde" AND tamaño = "grande" THEN fruta = "sandía"  
6. IF color = "verde" AND tamaño = "pequeño" THEN fruta = "kiwi"  


In [ ]:
def identificar_fruta(color, tamaño):
    # Reglas del sistema experto
    if color == "rojo" and tamaño == "pequeño":
        return "cereza"
    elif color == "amarillo" and tamaño == "grande":
        return "banana"
    elif color == "naranja" and tamaño == "mediano":
        return "mandarina"
    elif color == "rojo" and tamaño == "grande":
        return "manzana"
    elif color == "verde" and tamaño == "grande":
        return "sandía"
    elif color == "verde" and tamaño == "pequeño":
        return "kiwi"
    else:
        # Si no hay una regla para esa combinación
        return "No identificado - agrega una nueva regla"


#  Implementación en Python

def identificar_fruta(color, tamaño):
    # Reglas del sistema experto
    if color == "rojo" and tamaño == "pequeño":
        return "cereza"
    elif color == "amarillo" and tamaño == "grande":
        return "banana"
    elif color == "naranja" and tamaño == "mediano":
        return "mandarina"
    elif color == "rojo" and tamaño == "grande":
        return "manzana"
    elif color == "verde" and tamaño == "grande":
        return "sandía"
    elif color == "verde" and tamaño == "pequeño":
        return "kiwi"
    else:
        return "No identificado - agrega una nueva regla"

# Pruebas automáticas
casos = [
    ("rojo", "pequeño"),
    ("amarillo", "grande"),
    ("naranja", "mediano"),
    ("rojo", "grande"),
    ("verde", "grande"),
    ("verde", "pequeño"),
    ("morado", "pequeño")  # caso no contemplado
]

for i, (color, tamaño) in enumerate(casos, 1):
    print(f"Fruta {i}: color={color}, tamaño={tamaño} → {identificar_fruta(color, tamaño)}")


**Pruebas automáticas**

In [ ]:
casos = [
    ("rojo", "pequeño"),   # debería ser cereza
    ("amarillo", "grande"),# banana
    ("naranja", "mediano"),# mandarina
    ("rojo", "grande"),    # manzana
    ("verde", "grande"),   # sandía
    ("verde", "pequeño"),  # kiwi
    ("morado", "pequeño")  # no identificado
]

for i, (color, tamaño) in enumerate(casos, 1):
    print(f"Fruta {i}: color={color}, tamaño={tamaño} → {identificar_fruta(color, tamaño)}")


**Resultado que se obtiene**

Fruta 1: color=rojo, tamaño=pequeño → cereza
Fruta 2: color=amarillo, tamaño=grande → banana
Fruta 3: color=naranja, tamaño=mediano → mandarina
Fruta 4: color=rojo, tamaño=grande → manzana
Fruta 5: color=verde, tamaño=grande → sandía
Fruta 6: color=verde, tamaño=pequeño → kiwi
Fruta 7: color=morado, tamaño=pequeño → No identificado - agrega una nueva regla


#  Actividad

# 1. Agrega aca nuevas reglas IF... THEN... (por ejemplo: "morado, pequeño → uva")
# 2. Proba con nuevas combinaciones de color y tamaño.

def identificar_fruta_extendida(color, tamaño):
    if color == "rojo" and tamaño == "pequeño":
        return "cereza"
    elif color == "amarillo" and tamaño == "grande":
        return "banana"
    elif color == "naranja" and tamaño == "mediano":
        return "mandarina"
    elif color == "rojo" and tamaño == "grande":
        return "manzana"
    elif color == "verde" and tamaño == "grande":
        return "sandía"
    elif color == "verde" and tamaño == "pequeño":
        return "kiwi"
    
    # 💡 Nueva regla ejemplo
    elif color == "morado" and tamaño == "pequeño":
        return "uva"

    else:
        return "No identificado - agrega una nueva regla"

# Proba con tus propias combinaciones
print(identificar_fruta_extendida("morado", "pequeño"))
print(identificar_fruta_extendida("amarillo", "mediano"))
